# 07 生成模块与提示词工程第一部分：Context Assembler + Prompting 演示

> 本 notebook 对应 `schedule.md` 阶段 4（C0–C5）。
>
> 目标：
> 1. 从阶段 06 的检索结果读取候选文档块；
> 2. 用阶段 07 的 `ContextAssembler` 完成去重/多样化/控长组装；
> 3. 渲染四阶段医学提示模板；
> 4. 导出样例 JSON，供后续联调与回归使用。

---

## 你将看到什么

- **C0**：加载上游样例数据与模块（06 + 07）。
- **C1**：执行上下文组装，观察 metadata。
- **C2**：看“原始候选 vs 去重后 vs 多样化后”的差异。
- **C3**：看 token 上限对上下文长度和内容的影响。
- **C4**：将组装结果渲染成四阶段 Prompt（不调用 LLM）。
- **C5**：导出 `assembled_context_examples.json` 与 `prompt_examples.json`。

> 说明：本 notebook 默认使用 06 的离线样例 `pipeline_eval.json`，因此不依赖在线检索或大体积向量库。

## C0：加载数据与模块

这一步做三件事：

1. **定位目录**：确定 06（检索输出）和 07（本阶段代码）路径；
2. **导入模块**：`ContextAssembler` 与 `PROMPT_STAGES`；
3. **加载样例数据**：读取 06 的 `pipeline_eval.json`（样本库结果）。

> 为什么用离线样例？
>
> - 可复现：不依赖当下检索服务状态；
> - 轻量：无需重新连 Chroma/BM25；
> - 结构一致：与真实 `result["reranked"]` 字段相同。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# 项目根目录（当前 notebook 位于 07/notebooks）
ROOT = Path.cwd().resolve().parent.parent
STAGE06 = ROOT / "06 检索系统开发第二部分"
STAGE07 = ROOT / "07 生成模块与提示词工程第一部分"

# 将 07/src 加入路径，导入本阶段模块
sys.path.insert(0, str(STAGE07 / "src"))

from context_assembler import ContextAssembler
from prompts import PROMPT_STAGES, render_prompt_stage, validate_prompt_stage

PIPELINE_EVAL_PATH = STAGE06 / "outputs" / "samples" / "pipeline_eval.json"
with PIPELINE_EVAL_PATH.open("r", encoding="utf-8") as f:
    pipeline_eval = json.load(f)

print("ROOT:", ROOT)
print("Loaded:", PIPELINE_EVAL_PATH)
print("Query count:", pipeline_eval["query_count"])
print("Available prompt stages:", list(PROMPT_STAGES.keys()))

ROOT: D:\谷歌
Loaded: D:\谷歌\06 检索系统开发第二部分\outputs\samples\pipeline_eval.json
Query count: 5
Available prompt stages: ['evidence_evaluator', 'answer_generator', 'critical_reviewer', 'final_assembler']


## C1：ContextAssembler 组装 + metadata 展示

这一步对应任务书中的“上下文组装器核心产出”。

输入：`queries[0]["reranked"]`（以 `metformin cardiovascular effects` 为例）  
输出：

- `context_text`：给 LLM 的上下文正文；
- `metadata`：组装过程统计（候选数、去重后数、入选数、估算 token、来源分布）；
- `selected_chunks`：最终入选块（可追溯证据）。

> 这里先用 `tokenizer_name=None`，采用启发式 token 估算，避免首次运行触发 tokenizer 下载。

In [2]:
query_obj = pipeline_eval["queries"][0]
question = query_obj["query"]
retrieved_candidates = query_obj["reranked"]

assembler = ContextAssembler(tokenizer_name=None)
assembled = assembler.assemble(retrieved_candidates, max_context_tokens=1200)

print("Question:", question)
print("total_chunks_retrieved:", assembled.metadata.total_chunks_retrieved)
print("unique_chunks_after_dedup:", assembled.metadata.unique_chunks_after_dedup)
print("chunks_selected:", assembled.metadata.chunks_selected)
print("estimated_tokens:", assembled.metadata.estimated_tokens)
print("chunk_sources:", assembled.metadata.chunk_sources)
print("-" * 80)
print("Context preview:")
print(assembled.context_text[:700] + ("..." if len(assembled.context_text) > 700 else ""))

Question: metformin cardiovascular effects
total_chunks_retrieved: 5
unique_chunks_after_dedup: 5
chunks_selected: 3
estimated_tokens: 1200
chunk_sources: {'counts': {'PMC520826': 1, 'PMC521687': 1, 'PMC523844': 1}, 'unique_sources': 3}
--------------------------------------------------------------------------------
Context preview:
Daily rhythms in plasma levels of homocysteine

There is accumulated evidence that plasma concentration of the sulfur-containing amino-acid homocysteine (Hcy) is a prognostic marker for cardiovascular morbidity and mortality. Both fasting levels of Hcy and post methionine loading levels are used as prognostic markers. The aim of the present study was to investigate the existence of a daily rhythm in plasma Hcy under strictly controlled nutritional and sleep-wake conditions. We also investigated if the time during which methionine loading is performed, i.e., morning or evening, had a different effect on the resultant plasma Hcy concentration.
Six healthy men

## C2：去重 / 多样化前后对比

目的：让你直观看到 `ContextAssembler` 中两类策略的作用：

1. **去重（Jaccard）**：去掉内容过于重复的候选；
2. **多样化排序**：在相关性优先基础上，避免上下文被同一来源“刷屏”。

比较口径：

- 原始候选：06 `reranked`
- 去重后：`dedup_by_jaccard`
- 多样化后：`_order_with_diversity`（内部策略，演示用）

In [3]:
from models import coerce_to_document_chunks

chunks, skipped = coerce_to_document_chunks(retrieved_candidates)
unique_chunks = assembler.dedup_by_jaccard(chunks)
diverse_chunks = assembler._order_with_diversity(unique_chunks)  # 演示用

print("raw:", len(chunks), "| skipped_invalid:", skipped)
print("after_dedup:", len(unique_chunks))
print("after_diversity_order:", len(diverse_chunks))

print("\nTop-5 raw chunk_ids:")
print([c.chunk_id for c in chunks[:5]])

print("\nTop-5 after dedup chunk_ids:")
print([c.chunk_id for c in unique_chunks[:5]])

print("\nTop-5 after diversity order (chunk_id, source, score):")
for c in diverse_chunks[:5]:
    print((c.chunk_id, c.source[:70], round(c.relevance_score, 6)))

raw: 5 | skipped_invalid: 0
after_dedup: 5
after_diversity_order: 5

Top-5 raw chunk_ids:
['PMC520826', 'PMC521687', 'PMC523844', 'PMC523838_chunk2', 'PMC524175']

Top-5 after dedup chunk_ids:
['PMC520826', 'PMC521687', 'PMC523844', 'PMC523838_chunk2', 'PMC524175']

Top-5 after diversity order (chunk_id, source, score):
('PMC520826', 'Daily rhythms in plasma levels of homocysteine', 0.298591)
('PMC521687', 'Factors influencing preoperative stress response in coronary artery by', 0.280393)
('PMC523844', 'Distribution of Major Health Risks: Findings from the Global Burden of', 0.267694)
('PMC523838_chunk2', 'Nevirapine and Efavirenz Elicit Different Changes in Lipid Profiles in', 0.247805)
('PMC524175', 'Single nucleotide polymorphisms in the apolipoprotein B and low densit', 0.215493)


## C3：截断策略演示（token 限制 + 句号边界）

目的：验证“上下文长度受控”且“尽量在自然句子边界结束”。

做法：设置不同 `max_context_tokens`，观察：

- 最终 `estimated_tokens` 是否不超限；
- `context_text` 尾部是否较自然（优先在末 10% 找 `.`/`!`/`?`）。

> 提醒：若末 10% 区间本身没有句号，策略会退化为硬截断，这是设计上的可接受行为。

In [4]:
for limit in [300, 600, 1200]:
    out = assembler.assemble(retrieved_candidates, max_context_tokens=limit)
    text = out.context_text
    tail = text[-120:] if len(text) > 120 else text
    print(f"max_context_tokens={limit}")
    print("  estimated_tokens:", out.metadata.estimated_tokens)
    print("  chunks_selected:", out.metadata.chunks_selected)
    print("  ends_with_punct:", text.endswith((".", "!", "?")))
    print("  tail:", repr(tail))
    print()

max_context_tokens=300
  estimated_tokens: 286
  chunks_selected: 1
  ends_with_punct: True
  tail: 'wo experiments, subjects received every 3 hours 150 ml of specially designed low-protein liquid food (Ensure ® formula).'

max_context_tokens=600
  estimated_tokens: 600
  chunks_selected: 2
  ends_with_punct: False
  tail: 'ce it has been shown that even short-lasting sympathetic cardiovascular stimulation may have detrimental effects on pati'

max_context_tokens=1200
  estimated_tokens: 1200
  chunks_selected: 3
  ends_with_punct: False
  tail: 'increase cost-effectiveness.\nFor 26 selected risk factors, expert working groups conducted comprehensive reviews of data'



## C4：四阶段 Prompt 渲染示例（不调用 LLM）

这一步展示“提示工程模板”的直接使用方式：

- 输入：`question` + `assembled.context_text` + 通用约束 + 输出格式要求；
- 输出：四个阶段各自的 `system_prompt`、`user_prompt`、`temperature`、`max_tokens`。

你可以把这里的 payload 直接对接后续 LLM 调用层（LangChain/Ollama）。

In [5]:
constraints = (
    "Use only evidence in context. Do not fabricate citations. "
    "State uncertainty when evidence is weak or conflicting."
)
output_format = (
    "JSON with keys: answer, evidence_points, uncertainty, safety_note"
)

missing_report = {
    key: validate_prompt_stage(stage)
    for key, stage in PROMPT_STAGES.items()
}
print("Placeholder validation:", missing_report)

rendered_prompts = {
    key: render_prompt_stage(
        key,
        question=question,
        context=assembled.context_text,
        constraints=constraints,
        output_format=output_format,
    )
    for key in PROMPT_STAGES
}

for key, payload in rendered_prompts.items():
    print("=" * 100)
    print(key, "->", payload["stage"], "| temp=", payload["temperature"], "| max_tokens=", payload["max_tokens"])
    print("System:", payload["system_prompt"])
    preview = payload["user_prompt"][:600]
    print("User preview:\n", preview + ("..." if len(payload["user_prompt"]) > 600 else ""))

Placeholder validation: {'evidence_evaluator': [], 'answer_generator': [], 'critical_reviewer': [], 'final_assembler': []}
evidence_evaluator -> 证据评估器 | temp= 0.1 | max_tokens= 900
System: You are a medical evidence evaluator. Assess evidence quality and consistency conservatively.
User preview:
 Question:
metformin cardiovascular effects

Retrieved Context:
Daily rhythms in plasma levels of homocysteine

There is accumulated evidence that plasma concentration of the sulfur-containing amino-acid homocysteine (Hcy) is a prognostic marker for cardiovascular morbidity and mortality. Both fasting levels of Hcy and post methionine loading levels are used as prognostic markers. The aim of the present study was to investigate the existence of a daily rhythm in plasma Hcy under strictly controlled nutritional and sleep-wake conditions. We also investigated if the time during which methionine l...
answer_generator -> 答案生成器 | temp= 0.2 | max_tokens= 1200
System: You are a cautious medical assist

## C5：导出样例 JSON

导出两份文件到 `07/outputs/samples/`：

1. `assembled_context_examples.json`
   - 记录输入问题、组装统计、上下文预览、入选 chunk 摘要；
2. `prompt_examples.json`
   - 记录四阶段 prompt 的参数与内容（可用于后续调 LLM）。

这两份文件可用于：

- 回归测试（结构是否稳定）；
- 给后续 LangChain 链路做契约输入；
- 向老师/同学演示“检索结果如何进入生成前处理”。

In [6]:
OUTPUT_DIR = STAGE07 / "outputs" / "samples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assembled_example = {
    "question": question,
    "input_source": str(PIPELINE_EVAL_PATH),
    "input_candidate_count": len(retrieved_candidates),
    "assembled": assembled.to_dict(),
    "context_preview": assembled.context_text[:1200],
    "selected_chunk_brief": [
        {
            "chunk_id": c.chunk_id,
            "source": c.source,
            "relevance_score": c.relevance_score,
            "doc_id": c.metadata.get("doc_id"),
        }
        for c in assembled.selected_chunks
    ],
}

prompt_example = {
    "question": question,
    "constraints": constraints,
    "output_format": output_format,
    "stages": rendered_prompts,
}

assembled_out = OUTPUT_DIR / "assembled_context_examples.json"
prompt_out = OUTPUT_DIR / "prompt_examples.json"

with assembled_out.open("w", encoding="utf-8") as f:
    json.dump(assembled_example, f, ensure_ascii=False, indent=2)

with prompt_out.open("w", encoding="utf-8") as f:
    json.dump(prompt_example, f, ensure_ascii=False, indent=2)

print("Exported:")
print(" -", assembled_out)
print(" -", prompt_out)

Exported:
 - D:\谷歌\07 生成模块与提示词工程第一部分\outputs\samples\assembled_context_examples.json
 - D:\谷歌\07 生成模块与提示词工程第一部分\outputs\samples\prompt_examples.json
